# Rule-Based Stylometric Authorship Attribution

This notebook wraps the authorship pipeline into a notebook workflow for the ENG 683 coding project. The project stays symbolic and interpretable by using function-word frequencies, Burrows's Delta, POS trigram profiles, and lexical-richness diagnostics.

## Workflow

1. Load either a local corpus or a built-in NLTK demo corpus.
2. Chunk the texts into comparable samples.
3. Train author profiles from the training chunks.
4. Evaluate held-out chunks with Burrows's Delta as the main attribution score.
5. Inspect ranked candidate authors for each unknown chunk.

In [1]:
from collections import Counter
from pathlib import Path

from authorship_pipeline import (
    SourceDocument,
    ensure_nltk_resources,
    evaluate_model,
    load_local_corpus,
    prepare_chunks,
    split_chunks_by_author,
    train_model,
)

In [2]:
USE_LOCAL_CORPUS = False
LOCAL_CORPUS_DIR = Path("data/corpus")
CHUNK_SIZE = 1500
MIN_CHUNK_SIZE = 900
HOLDOUT_PER_AUTHOR = 1
FUNCTION_WORD_FEATURES = 40
POS_TRIGRAM_FEATURES = 30
TOP_RESULTS = 3

In [3]:
def load_demo_documents():
    from nltk.corpus import gutenberg

    return [
        SourceDocument(
            author="Jane Austen",
            title="Sense and Sensibility",
            text=gutenberg.raw("austen-sense.txt"),
        ),
        SourceDocument(
            author="G. K. Chesterton",
            title="The Man Who Was Thursday",
            text=gutenberg.raw("chesterton-thursday.txt"),
        ),
        SourceDocument(
            author="Maria Edgeworth",
            title="Parents' Assistant",
            text=gutenberg.raw("edgeworth-parents.txt"),
        ),
    ]


def summarize_chunks(chunks):
    counts = Counter(chunk.author for chunk in chunks)
    return {author: counts[author] for author in sorted(counts)}


def build_result_rows(summary, top_results=3):
    rows = []
    for chunk_score in summary.chunk_scores:
        for rank, result in enumerate(chunk_score.ranking[:top_results], start=1):
            rows.append(
                {
                    "true_author": chunk_score.chunk.author,
                    "title": chunk_score.chunk.title,
                    "chunk_index": chunk_score.chunk.chunk_index,
                    "predicted_author": chunk_score.predicted_author,
                    "rank": rank,
                    "candidate_author": result.author,
                    "delta_score": round(result.delta_score, 4),
                    "pos_similarity": round(result.pos_similarity, 4),
                    "lexical_distance": round(result.lexical_distance, 4),
                }
            )
    return rows

In [4]:
ensure_nltk_resources(include_gutenberg=not USE_LOCAL_CORPUS)

if USE_LOCAL_CORPUS:
    documents = load_local_corpus(LOCAL_CORPUS_DIR)
    corpus_source = f"Local corpus: {LOCAL_CORPUS_DIR}"
else:
    documents = load_demo_documents()
    corpus_source = "NLTK Gutenberg demo corpus"

print(corpus_source)
[(document.author, document.title) for document in documents]

NLTK Gutenberg demo corpus


[('Jane Austen', 'Sense and Sensibility'),
 ('G. K. Chesterton', 'The Man Who Was Thursday'),
 ('Maria Edgeworth', "Parents' Assistant")]

In [5]:
chunks = prepare_chunks(
    documents,
    chunk_size=CHUNK_SIZE,
    min_chunk_size=MIN_CHUNK_SIZE,
)

print("Prepared chunks by author:")
summarize_chunks(chunks)

Prepared chunks by author:


{'G. K. Chesterton': 39, 'Jane Austen': 81, 'Maria Edgeworth': 114}

In [6]:
training_chunks, test_chunks = split_chunks_by_author(
    chunks,
    holdout_per_author=HOLDOUT_PER_AUTHOR,
)

model = train_model(
    training_chunks,
    function_word_features=FUNCTION_WORD_FEATURES,
    pos_trigram_features=POS_TRIGRAM_FEATURES,
)

print(f"Training chunks: {len(training_chunks)}")
print(f"Test chunks: {len(test_chunks)}")
print(f"Function-word features kept: {len(model.function_words)}")
print(f"POS trigram features kept: {len(model.pos_trigrams)}")
model.function_words[:20]

Training chunks: 231
Test chunks: 3
Function-word features kept: 40
POS trigram features kept: 30


('the',
 'to',
 'and',
 'of',
 'a',
 'i',
 'in',
 'he',
 'you',
 'was',
 'it',
 'that',
 'her',
 'his',
 'for',
 'not',
 'as',
 'she',
 'with',
 'be')

In [7]:
summary = evaluate_model(model, test_chunks)
print(
    f"Holdout accuracy: {summary.correct_predictions}/{summary.total_chunks} = {summary.accuracy:.2%}"
)

Holdout accuracy: 3/3 = 100.00%


In [8]:
results = build_result_rows(summary, top_results=TOP_RESULTS)
results

[{'true_author': 'G. K. Chesterton',
  'title': 'The Man Who Was Thursday',
  'chunk_index': 38,
  'predicted_author': 'G. K. Chesterton',
  'rank': 1,
  'candidate_author': 'G. K. Chesterton',
  'delta_score': 0.7242,
  'pos_similarity': 0.9667,
  'lexical_distance': 1.6187},
 {'true_author': 'G. K. Chesterton',
  'title': 'The Man Who Was Thursday',
  'chunk_index': 38,
  'predicted_author': 'G. K. Chesterton',
  'rank': 2,
  'candidate_author': 'Maria Edgeworth',
  'delta_score': 0.7853,
  'pos_similarity': 0.9363,
  'lexical_distance': 3.833},
 {'true_author': 'G. K. Chesterton',
  'title': 'The Man Who Was Thursday',
  'chunk_index': 38,
  'predicted_author': 'G. K. Chesterton',
  'rank': 3,
  'candidate_author': 'Jane Austen',
  'delta_score': 0.9655,
  'pos_similarity': 0.908,
  'lexical_distance': 11.4017},
 {'true_author': 'Jane Austen',
  'title': 'Sense and Sensibility',
  'chunk_index': 80,
  'predicted_author': 'Jane Austen',
  'rank': 1,
  'candidate_author': 'Jane Austen

## Final Project Notes

You can replace the demo corpus with your own local corpus under `data/corpus/<author>/<title>.txt`. The strongest version of this project uses multiple texts per author in the same general genre and period, because authorship attribution is sensitive to topic, genre, and text length.

In [9]:
# Optional export cell
# Uncomment these lines if you want to save the ranked results to disk.
# from json import dumps
# results_path = Path("data/holdout_results.json")
# results_path.write_text(dumps(results, indent=2), encoding="utf-8")
# results_path